In [1]:
# ----------------------------------------------------------
# 1. SQL  (a) resultado exacto de la consulta
# ----------------------------------------------------------
import duckdb
import pandas as pd

ventas = pd.DataFrame({
    "pais":     ["GT", "SV", "GT", "HN", "GT", "SV"],
    "producto": ["cafe", "cafe", "banano", "cafe", "cafe", "banano"],
    "monto":    [100, 80, 50, 70, 90, 40],
})

resultado = duckdb.sql("""
    SELECT pais, SUM(monto) AS total
    FROM ventas
    GROUP BY pais
    HAVING SUM(monto) > 100
    ORDER BY total DESC
""")

resultado.show()

print("filas:", len(resultado.fetchall()), " columnas:", len(resultado.columns))

┌─────────┬────────┐
│  pais   │ total  │
│ varchar │ int128 │
├─────────┼────────┤
│ GT      │    240 │
│ SV      │    120 │
└─────────┴────────┘

filas: 2  columnas: 2


HAVING filtra grupos ya agregados —descarta los países que no superan 100—, a diferencia de WHERE, que filtraría filas individuales antes de agrupar.

In [2]:
# ----------------------------------------------------------
# 1. SQL  (b) cuantas ventas hay por producto
# ----------------------------------------------------------
por_producto = duckdb.sql("""
    SELECT producto, COUNT(*) AS ventas
    FROM ventas
    GROUP BY producto
    ORDER BY ventas DESC
""")

por_producto.show()

┌──────────┬────────┐
│ producto │ ventas │
│ varchar  │ int64  │
├──────────┼────────┤
│ cafe     │      4 │
│ banano   │      2 │
└──────────┴────────┘



In [3]:
# ----------------------------------------------------------
# 2. NAIVE BAYES con Laplace  (a)
# ----------------------------------------------------------
# P(palabra | clase) = (conteo + alpha) / (total + alpha * |V|)

alpha = 1
V = 4
total_spam = 6
total_normal = 6

oferta_spam, oferta_normal = 2, 0
clase_spam,  clase_normal  = 0, 3

den_spam   = total_spam   + alpha * V
den_normal = total_normal + alpha * V

p_oferta_spam   = (oferta_spam   + alpha) / den_spam
p_oferta_normal = (oferta_normal + alpha) / den_normal
p_clase_spam    = (clase_spam    + alpha) / den_spam
p_clase_normal  = (clase_normal  + alpha) / den_normal

print("Las 4 probabilidades suavizadas (denominador = 6 + 1*4 = 10):")
print("  P(oferta | spam)   = (2+1)/10 =", p_oferta_spam)
print("  P(oferta | normal) = (0+1)/10 =", p_oferta_normal)
print("  P(clase  | spam)   = (0+1)/10 =", p_clase_spam)
print("  P(clase  | normal) = (3+1)/10 =", p_clase_normal)

# Clasificar "oferta clase". Los priors son iguales: se cancelan.
score_spam   = p_oferta_spam   * p_clase_spam
score_normal = p_oferta_normal * p_clase_normal

print("\nLas dos cantidades que se comparan:")
print("  spam   = 0.3 * 0.1 =", round(score_spam, 4))
print("  normal = 0.1 * 0.4 =", round(score_normal, 4))
print("\nSe clasifica como:", "normal" if score_normal > score_spam else "spam")

Las 4 probabilidades suavizadas (denominador = 6 + 1*4 = 10):
  P(oferta | spam)   = (2+1)/10 = 0.3
  P(oferta | normal) = (0+1)/10 = 0.1
  P(clase  | spam)   = (0+1)/10 = 0.1
  P(clase  | normal) = (3+1)/10 = 0.4

Las dos cantidades que se comparan:
  spam   = 0.3 * 0.1 = 0.03
  normal = 0.1 * 0.4 = 0.04

Se clasifica como: normal


Sin suavizado ambos puntajes serían exactamente 0 —"clase" no aparece en spam y "oferta" no aparece en normal—, así que no habría con qué decidir; el +alpha de Laplace le da probabilidad no nula a lo no visto y permite que gane normal, empujada por las 3 apariciones de "clase".

In [4]:
# ----------------------------------------------------------
# 3. KNN  -  punto nuevo (4, 2)
# ----------------------------------------------------------
import math

nuevo = (4, 2)

datos = [
    ("p1", (3, 2), "A"),
    ("p2", (4, 4), "B"),
    ("p3", (6, 2), "B"),
    ("p4", (1, 1), "A"),
]

print("Distancias euclidianas desde (4, 2):")
distancias = []
for nombre, (x, y), clase in datos:
    dx = nuevo[0] - x
    dy = nuevo[1] - y
    d2 = dx**2 + dy**2
    d = math.sqrt(d2)
    distancias.append((d, nombre, clase))
    print(f"  {nombre} ({x},{y}) clase {clase}: raiz({dx**2} + {dy**2}) = raiz({d2}) = {d:.4f}")

distancias.sort()

# k = 1
print("\nk = 1 -> vecino mas cercano:", distancias[0][1], "-> clase", distancias[0][2])

# k = 3
vecinos_k3 = distancias[:3]
votos = {}
for d, nombre, clase in vecinos_k3:
    votos[clase] = votos.get(clase, 0) + 1

print("\nk = 3 -> votos:")
for clase, n in sorted(votos.items()):
    quienes = [nom for d, nom, c in vecinos_k3 if c == clase]
    print(f"  clase {clase}: {n}  {quienes}")

print("k = 3 -> clase", max(votos, key=votos.get))

Distancias euclidianas desde (4, 2):
  p1 (3,2) clase A: raiz(1 + 0) = raiz(1) = 1.0000
  p2 (4,4) clase B: raiz(0 + 4) = raiz(4) = 2.0000
  p3 (6,2) clase B: raiz(4 + 0) = raiz(4) = 2.0000
  p4 (1,1) clase A: raiz(9 + 1) = raiz(10) = 3.1623

k = 1 -> vecino mas cercano: p1 -> clase A

k = 3 -> votos:
  clase A: 1  ['p1']
  clase B: 2  ['p2', 'p3']
k = 3 -> clase B


Un k chico (k=1) sobreajusta y es frágil ante el ruido: un solo vecino mal etiquetado o atípico decide toda la predicción.

Un k grande sobresuaviza: incorpora vecinos lejanos e irrelevantes y la predicción se arrastra hacia la clase mayoritaria, borrando la estructura local.

Uso de IA e internet

Se le pasaron las tres preguntas del examen y resolvió lo siguiente:

Se pidio la sintaxis para utilizar duck db en las consultas sql
